# Nussinov RNA Folding

Team 9 micropresentation, CHEM 274B 


## A. Algorithm: filling the table



- fill_nussinov_table(rna_sequence, minimum_hairpin_loop_length=3) returns an N x N list of lists
- table[ i ][ j ] is the maximum number of base pairs in the subsequence from position i to j (0-indexed, inclusive)
- Only the upper triangle ( i < j ) is filled; all other cells stay 0
- The traceback in section B starts from table[ 0 ][ N-1 ]

In [1]:
# Base pairs Nussinov allows: Watson-Crick (A-U, G-C) and G-U wobble, in either order
allowed_base_pairs = {("A", "U"), ("U", "A"), ("G", "C"), ("C", "G"), ("G", "U"), ("U", "G")}

def can_bases_pair(first_base, second_base):
    # True if the two bases form one of the allowed pairs
    return (first_base, second_base) in allowed_base_pairs

In [2]:
print(can_bases_pair("A", "U"), can_bases_pair("U", "A"), can_bases_pair("G", "C"), can_bases_pair("G", "U"))
print(can_bases_pair("A", "G"), can_bases_pair("C", "U"))

True True True True
False False


In [8]:
def fill_nussinov_table(rna_sequence, minimum_hairpin_loop_length=3):
    # One row and one column per base in the sequence
    sequence_length = len(rna_sequence)
    # Every cell starts at 0 (no pairs found yet)
    maximum_base_pairs_table = [[0] * sequence_length for _ in range(sequence_length)]

    # Fill shorter pieces first, since every case reuses answers for shorter pieces
    # Ends closer than (hairpin loop + 1) cannot pair, so those cells stay 0
    for distance_between_ends in range(minimum_hairpin_loop_length + 1, sequence_length):
        for start_position in range(sequence_length - distance_between_ends):
            end_position = start_position + distance_between_ends

            # Case 1: the first base is left unpaired
            pairs_if_start_unpaired = maximum_base_pairs_table[start_position + 1][end_position]
            # Case 2: the last base is left unpaired
            pairs_if_end_unpaired = maximum_base_pairs_table[start_position][end_position - 1]
            # Case 3: the first and last bases pair with each other
            pairs_if_ends_pair = 0
            if can_bases_pair(rna_sequence[start_position], rna_sequence[end_position]):
                pairs_if_ends_pair = maximum_base_pairs_table[start_position + 1][end_position - 1] + 1
            # Case 4: split into two independent pieces, trying every split point
            pairs_if_split = 0
            for split_position in range(start_position + 1, end_position):
                pairs_in_both_pieces = maximum_base_pairs_table[start_position][split_position] + maximum_base_pairs_table[split_position + 1][end_position]
                pairs_if_split = max(pairs_if_split, pairs_in_both_pieces)

            maximum_base_pairs_table[start_position][end_position] = max(pairs_if_start_unpaired, pairs_if_end_unpaired, pairs_if_ends_pair, pairs_if_split)

    return maximum_base_pairs_table

In [9]:
for table_row in fill_nussinov_table("GGGAAAUCC"):
    print(table_row)
print("GGGAAAUCC whole sequence:", fill_nussinov_table("GGGAAAUCC")[0][8])
print("GAAACGAAAC whole sequence:", fill_nussinov_table("GAAACGAAAC")[0][9])

[0, 0, 0, 0, 0, 0, 1, 2, 3]
[0, 0, 0, 0, 0, 0, 1, 2, 2]
[0, 0, 0, 0, 0, 0, 1, 1, 1]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0]
GGGAAAUCC whole sequence: 3
GAAACGAAAC whole sequence: 2


## B. Structure output: traceback and display



## C. Biology: a real RNA example



## D. Analysis: timing and complexity

